In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive
from datetime import date
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

In [ ]:
drive.mount("/content/drive")
duong_dan_csv = ("/content/drive/MyDrive/car_price_project/Car details v3.csv")
#Doc thanh bang du lieu
df = pd.read_csv(duong_dan_csv)

print("Số hàng, số cột:", df.shape)
display(df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Số hàng, số cột: (8128, 13)


,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.4 kmpl,1248 CC,74 bhp,190Nm@ 2000rpm,5.0
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14 kmpl,1498 CC,103.52 bhp,250Nm@ 1500-2500rpm,5.0
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.7 kmpl,1497 CC,78 bhp,"12.7@ 2,700(kgm@ rpm)",5.0
3,Hyundai i20 Sportz Diesel,2010,225000,127000,Diesel,Individual,Manual,First Owner,23.0 kmpl,1396 CC,90 bhp,22.4 kgm at 1750-2750rpm,5.0
4,Maruti Swift VXI BSIII,2007,130000,120000,Petrol,Individual,Manual,First Owner,16.1 kmpl,1298 CC,88.2 bhp,"11.5@ 4,500(kgm@ rpm)",5.0


In [ ]:
df.info()

#Đếm số dòng thiếu
print("\nSố ô thiếu trong từng cột:")
print(df.isna().sum())

#Đếm số dòng trùng
print("\nSố dòng trùng:", df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8128 entries, 0 to 8127
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   name           8128 non-null   object 
 1   year           8128 non-null   int64  
 2   selling_price  8128 non-null   int64  
 3   km_driven      8128 non-null   int64  
 4   fuel           8128 non-null   object 
 5   seller_type    8128 non-null   object 
 6   transmission   8128 non-null   object 
 7   owner          8128 non-null   object 
 8   mileage        7907 non-null   object 
 9   engine         7907 non-null   object 
 10  max_power      7913 non-null   object 
 11  torque         7906 non-null   object 
 12  seats          7907 non-null   float64
dtypes: float64(1), int64(3), object(9)
memory usage: 825.6+ KB

Số ô thiếu trong từng cột:
name               0
year               0
selling_price      0
km_driven          0
fuel               0
seller_type        0
transmission   

In [ ]:
df_clean = df.copy()
initial_row = len(df_clean)
df_clean = df_clean.drop_duplicates().reset_index(drop = True)
last_row = len(df_clean)

print("Trước xử lý:", initial_row)
print("Sau xử lý", last_row)
print("Đã loại bỏ", initial_row - last_row)

Trước xử lý: 8128
Sau xử lý 6926
Đã loại bỏ 1202


In [ ]:
#Chuyển đổi engine thành số đếm
engine_text = (df_clean["engine"].str.replace("CC", "", regex=False).str.strip())
df_clean["engine_cc"] = pd.to_numeric(engine_text, errors = "coerce")

#Đối chiếu trước và sau
display(df_clean[["engine", "engine_cc"]].head(10))

#Kiểm tra kết quả
print("Kiểu dữ liệu:", df_clean["engine_cc"].dtype)
print("Số ô thiếu:", df_clean["engine_cc"].isna().sum())

,engine,engine_cc
0,1248 CC,1248.0
1,1498 CC,1498.0
2,1497 CC,1497.0
3,1396 CC,1396.0
4,1298 CC,1298.0
5,1197 CC,1197.0
6,1061 CC,1061.0
7,796 CC,796.0
8,1364 CC,1364.0
9,1399 CC,1399.0


Kiểu dữ liệu: float64
Số ô thiếu: 208


In [ ]:
#Chuyển đổi max_power thành số đếm
power_text = (
    df_clean["max_power"]
    .str.replace("bhp", "", regex=False)
    .str.strip()
)
df_clean["max_power_bhp"] = pd.to_numeric(
    power_text,
    errors="coerce"
)

# Đối chiếu trước và sau
display(df_clean[["max_power", "max_power_bhp"]].head(10))

#Kiểm tra kết quả
print("Kiểu dữ liệu:", df_clean["max_power_bhp"].dtype)
print("Số ô thiếu:", df_clean["max_power_bhp"].isna().sum())

,max_power,max_power_bhp
0,74 bhp,74.00
1,103.52 bhp,103.52
2,78 bhp,78.00
3,90 bhp,90.00
4,88.2 bhp,88.20
5,81.86 bhp,81.86
6,57.5 bhp,57.50
7,37 bhp,37.00
8,67.1 bhp,67.10
9,68.1 bhp,68.10


Kiểu dữ liệu: float64
Số ô thiếu: 206


In [ ]:
#Đối chiếu các cột xem có dữ liệu nào khác nhau
cac_cot_chu = [
    "fuel",
    "seller_type",
    "transmission",
    "owner"
]

for cot in cac_cot_chu:
    print("Cột:", cot)
    print(df_clean[cot].unique())
    print()

Cột: fuel
['Diesel' 'Petrol' 'LPG' 'CNG']

Cột: seller_type
['Individual' 'Dealer' 'Trustmark Dealer']

Cột: transmission
['Manual' 'Automatic']

Cột: owner
['First Owner' 'Second Owner' 'Third Owner' 'Fourth & Above Owner'
 'Test Drive Car']



In [ ]:

cac_cot_so = [
    "year",
    "selling_price",
    "km_driven",
    "engine_cc",
    "max_power_bhp",
    "seats"
]

display(df_clean[cac_cot_so].describe().T.round(2))

,count,mean,std,min,25%,50%,75%,max
year,6926.0,2013.42,4.08,1983.0,2011.0,2014.00,2017.0,2020.0
selling_price,6926.0,517270.68,519766.99,29999.0,250000.0,400000.00,633500.0,10000000.0
km_driven,6926.0,73995.68,58358.10,1.0,40000.0,70000.00,100000.0,2360457.0
engine_cc,6718.0,1430.89,493.49,624.0,1197.0,1248.00,1498.0,3604.0
max_power_bhp,6720.0,87.73,31.77,0.0,67.1,81.83,100.0,400.0
seats,6718.0,5.43,0.98,2.0,5.0,5.00,5.0,14.0


In [ ]:
#Kiểm tra dữ liệu xem dữ liệu số có vi phạm gì không
quy_tac = {
    "Giá không dương":
        df_clean["selling_price"] <= 0,

    "Số km âm":
        df_clean["km_driven"] < 0,

    "Số km không nguyên":
        df_clean["km_driven"].notna()
        & (df_clean["km_driven"] % 1 != 0),

    "Dung tích động cơ không dương":
        df_clean["engine_cc"] <= 0,

    "Công suất không dương":
        df_clean["max_power_bhp"] <= 0,

    "Số ghế không dương":
        df_clean["seats"] <= 0,

    "Số ghế không nguyên":
        df_clean["seats"].notna()
        & (df_clean["seats"] % 1 != 0),

    "Năm không dương hoặc vượt năm hiện tại":
        (df_clean["year"] <= 0)
        | (df_clean["year"] > date.today().year),

    "Năm không nguyên":
        df_clean["year"].notna()
        & (df_clean["year"] % 1 != 0)
}

for ten_quy_tac, dieu_kien in quy_tac.items():
    print(ten_quy_tac, ":", dieu_kien.sum(), "dòng")

Giá không dương : 0 dòng
Số km âm : 0 dòng
Số km không nguyên : 0 dòng
Dung tích động cơ không dương : 0 dòng
Công suất không dương : 3 dòng
Số ghế không dương : 0 dòng
Số ghế không nguyên : 0 dòng
Năm không dương hoặc vượt năm hiện tại : 0 dòng
Năm không nguyên : 0 dòng


In [ ]:
#Kiểm tra công suất không dương
dieu_kien = df_clean["max_power_bhp"] <= 0

display(
    df_clean.loc[
        dieu_kien,
        [
            "name",
            "fuel",
            "engine_cc",
            "max_power",
            "max_power_bhp"
        ]
    ]
)

# Chuyển riêng các ô công suất đó thành giá trị thiếu
df_clean.loc[dieu_kien, "max_power_bhp"] = np.nan

#Kiểm tra lại
print(
    "Số công suất không dương:",
    (df_clean["max_power_bhp"] <= 0).sum()
)

print(
    "Số ô công suất thiếu:",
    df_clean["max_power_bhp"].isna().sum()
)

print("Số dòng còn lại:", len(df_clean))

,name,fuel,engine_cc,max_power,max_power_bhp
547,Maruti Alto K10 LXI,Petrol,NaN,0,0.0
1309,Maruti Swift Dzire VDI Optional,Diesel,NaN,0,0.0
2352,Tata Indica Vista Quadrajet LS,Diesel,NaN,0,0.0


Số công suất không dương: 0
Số ô công suất thiếu: 209
Số dòng còn lại: 6926


In [ ]:
#Chọn các cột làm đầu vào
cot_so = [
    "year",
    "km_driven",
    "engine_cc",
    "max_power_bhp",
    "seats"
]

cot_chu = ["fuel", "transmission", "owner"]

#Input và Output
X = df_clean[cot_so + cot_chu].copy()
y = df_clean["selling_price"].copy()

#Chia train và test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
print("Số mẫu train:", len(X_train))
print("Số mẫu test:", len(X_test))

Số mẫu train: 5540
Số mẫu test: 1386


In [ ]:
#Tính trung vị riêng cho mỗi cột
cot_dien_trung_vi = ["engine_cc", "max_power_bhp"]
trung_vi_train = X_train[cot_dien_trung_vi].median()

print("Giá trị dùng để điền:")
print(trung_vi_train)

# Áp dụng cùng các giá trị đó cho cả hai tập
X_train[cot_dien_trung_vi] = (
    X_train[cot_dien_trung_vi].fillna(trung_vi_train)
)

X_test[cot_dien_trung_vi] = (
    X_test[cot_dien_trung_vi].fillna(trung_vi_train)
)

Giá trị dùng để điền:
engine_cc        1248.00
max_power_bhp      81.83
dtype: float64


In [ ]:
# Lấy số ghế phổ biến nhất trên train
so_ghe_pho_bien = X_train["seats"].mode().iloc[0]

print("Số ghế dùng để điền:", so_ghe_pho_bien)

X_train["seats"] = X_train["seats"].fillna(so_ghe_pho_bien)
X_test["seats"] = X_test["seats"].fillna(so_ghe_pho_bien)

Số ghế dùng để điền: 5.0


In [ ]:
#Kiểm tra xem đã đủ bộ train và test chưa
print("Ô thiếu trên train:")
print(X_train.isna().sum())

print("\nÔ thiếu trên test:")
print(X_test.isna().sum())

Ô thiếu trên train:
year             0
km_driven        0
engine_cc        0
max_power_bhp    0
seats            0
fuel             0
transmission     0
owner            0
dtype: int64

Ô thiếu trên test:
year             0
km_driven        0
engine_cc        0
max_power_bhp    0
seats            0
fuel             0
transmission     0
owner            0
dtype: int64


In [ ]:
# Tạo bộ mã hóa
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

# Học các nhóm từ train rồi mã hóa train
train_chu_encoded = encoder.fit_transform(X_train[cot_chu])

# Mã hóa test theo đúng quy tắc đã học từ train
test_chu_encoded = encoder.transform(X_test[cot_chu])

# Lấy tên các cột mới
ten_cot_moi = encoder.get_feature_names_out(cot_chu)

# Chuyển kết quả thành bảng pandas
train_chu_df = pd.DataFrame(
    train_chu_encoded,
    columns=ten_cot_moi,
    index=X_train.index
)

test_chu_df = pd.DataFrame(
    test_chu_encoded,
    columns=ten_cot_moi,
    index=X_test.index
)

# Ghép các cột số với các cột vừa mã hóa
X_train_encoded = pd.concat(
    [X_train[cot_so], train_chu_df],
    axis=1
)

X_test_encoded = pd.concat(
    [X_test[cot_so], test_chu_df],
    axis=1
)

print("Kích thước train sau mã hóa:", X_train_encoded.shape)
print("Kích thước test sau mã hóa:", X_test_encoded.shape)

display(X_train_encoded.head())

Kích thước train sau mã hóa: (5540, 16)
Kích thước test sau mã hóa: (1386, 16)


,year,km_driven,engine_cc,max_power_bhp,seats,fuel_CNG,fuel_Diesel,fuel_LPG,fuel_Petrol,transmission_Automatic,transmission_Manual,owner_First Owner,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
4855,2018,25000,1498.0,98.96,5.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
3143,2010,170000,1248.0,81.83,5.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
4275,2011,75500,1396.0,90.00,5.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
2288,2009,138000,1248.0,73.90,5.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
3914,2003,200000,1493.0,82.00,5.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0


In [ ]:
# Tạo mô hình
model = LinearRegression()

# Huấn luyện bằng thông tin xe và giá tương ứng
model.fit(X_train_encoded, y_train)

print("Đã huấn luyện xong!")

Đã huấn luyện xong!


In [ ]:
# Dự đoán giá của 1386 xe trong tập test
y_pred = model.predict(X_test_encoded)

# Đặt giá thực tế và giá dự đoán cạnh nhau
ket_qua = pd.DataFrame({
    "gia_thuc_te": y_test,
    "gia_du_doan": y_pred
})

ket_qua["sai_lech_tuyet_doi"] = (
    ket_qua["gia_thuc_te"] - ket_qua["gia_du_doan"]
).abs()

display(
    ket_qua.head(10).style.format("{:,.0f}")
)

,gia_thuc_te,gia_du_doan,sai_lech_tuyet_doi
6878,"550,000","400,307","149,693"
3660,"1,225,000","1,111,911","113,089"
5604,"850,000","791,649","58,351"
3806,"80,000","-17,855","97,855"
2338,"825,000","839,949","14,949"
6028,"800,000","855,358","55,358"
2812,"229,999","361,099","131,100"
1320,"93,150","81,497","11,653"
6496,"300,000","246,044","53,956"
2992,"493,000","566,513","73,513"


In [ ]:
#Đánh giá mô hình
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("R²:", r2)

MAE: 171214.2440750507
R²: 0.5460945400259307
